In [1]:
# This allows us to import from all folders one level up from notebooks folder - run 1 time
import sys
from pathlib import Path

print('All paths pre-append')
for i,p in enumerate(sys.path):
    print(f"{i}: {p}")
print('-'*100)

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
print('All paths post-append')
for i,p in enumerate(sys.path):
    print(f"{i}: {p}")


All paths pre-append
0: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python311.zip
1: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11
2: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11/lib-dynload
3: 
4: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11/site-packages
----------------------------------------------------------------------------------------------------
All paths post-append
0: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python311.zip
1: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11
2: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11/lib-dynload
3: 
4: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11/site-packages
5: /Users/irabandutta/Developer/2026-08-llm-from-scratch


# Imports

In [2]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from typing import Tuple
from src.model.llm_config import LLMConfig
from src.model.llm import LLM

# DataLoader

In [3]:
class TokenDataLoader:
    def __init__(self, B:int, T:int, binary_file_path:str, dtype:np.dtype, debug:bool=False):
        self.B = max(1, B)
        self.T = max(4, T)
        if not debug:
            self.tokens = np.memmap(
                filename=binary_file_path,
                dtype=dtype,
                mode='r'
            )
            print(f"Loaded {len(self.tokens)/10e6}M tokens, 1 epoch = {len(self.tokens)//(self.T)} batches")
        else:
            self.tokens = np.arange(1, 101)
        if len(self.tokens)<=(self.B)*(self.T):
            raise ValueError(
                f"Current values of batch size {B} and seq length {T} are too large for dataset, please reduce either or both"
            )
        self.curr_idx = 0

    def next_batch(self) -> Tuple[torch.tensor]:
        B, T = (self.B), (self.T)

        buffer = self.tokens[self.curr_idx:self.curr_idx+(B*T+1)]
        x = torch.tensor(buffer[:-1]).view(B, T)
        y = torch.tensor(buffer[1:]).view(B, T)

        # Covert x,y from uint16 to int32 and int64
        x = x.int()
        y = y.long()

        self.curr_idx += B*T
        if self.curr_idx+(B*T+1) > len(self.tokens):
            self.curr_idx=0

        return x, y


binary_file_path = '../data/tinystories/processed/train.bin'
B = 4
T = 32
tok_dl = TokenDataLoader(B, T, binary_file_path, np.uint16, False)
# print('-'*100)
# steps = 2
# for _ in range(steps):
#     x, y = tok_dl.next_batch()
#     print('x:\n', x)
#     print('y:\n', y)    
# print('-'*100)

Loaded 47.1872517M tokens, 1 epoch = 14746016 batches


In [4]:
# Get a sample batch of tokens of shape (B, T) and overfit model on that batch

x, y = tok_dl.next_batch()
print(x.shape, y.shape)
print(x.dtype, y.dtype)

torch.Size([4, 32]) torch.Size([4, 32])
torch.int32 torch.int64


# Instantiate model

In [5]:
# ======== DEFINE Model ========
ctx_len = T
d_model = 256

llm_config = LLMConfig(
    vocab_size=50257,
    ctx_len=ctx_len,
    d_model=d_model, 
    n_layer=4,
    ff_ratio=4,
    dropout=0.0,
    eps=1e-5,
    position_embedding='sinusoidal',
    rotary_embedding=False,
    attention='mha',
    normalization='layernorm',
    n_heads=4, 
    n_groups=None,
    use_flash=False, 
    attn_debug=False
)
print(llm_config)
print('-'*50)

# Detect and resolve device
device = 'cpu'
if torch.cuda.is_available():
    device='cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device='mps'

print("Device found:", device)
print('-'*50)


# Instantiate model
model = LLM(llm_config)
model.eval()
model = model.to(device)
print(f"Model instantiated and moved to {device}")
print('-'*50)

LLMConfig(vocab_size=50257, ctx_len=32, d_model=256, n_layer=4, ff_ratio=4, dropout=0.0, eps=1e-05, position_embedding='sinusoidal', rotary_embedding=False, attention='mha', normalization='layernorm', n_heads=4, n_groups=None, use_flash=False, attn_debug=False)
--------------------------------------------------
Device found: mps
--------------------------------------------------
Model instantiated and moved to mps
--------------------------------------------------


In [6]:
# Move x and y to device
print(x.device, y.device)
x, y = x.to(device), y.to(device)
print(x.device, y.device)
print(x.dtype, y.dtype)

cpu cpu
mps:0 mps:0
torch.int32 torch.int64


In [7]:
# 1 forward pass - track init loss
logits, loss = model(x, y)
print(logits.shape)
print(loss)

torch.Size([4, 32, 50257])
tensor(11.0068, device='mps:0', grad_fn=<NllLossBackward0>)


In [8]:
# Expected Loss at inti
-torch.log(torch.tensor(1/50257))

tensor(10.8249)

In [9]:
# Run a train loop over sample batch

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

steps = 50
for i in range(steps):
    optimizer.zero_grad()
    logits, loss = model(x, y)
    loss.backward()
    optimizer.step()
    print(f"Step: {i+1}: Loss: {loss.item()}")



Step: 1: Loss: 11.006792068481445
Step: 2: Loss: 10.390134811401367
Step: 3: Loss: 10.146204948425293
Step: 4: Loss: 9.896349906921387
Step: 5: Loss: 9.747297286987305
Step: 6: Loss: 9.593015670776367
Step: 7: Loss: 9.493810653686523
Step: 8: Loss: 9.37998104095459
Step: 9: Loss: 9.297237396240234
Step: 10: Loss: 9.19422721862793
Step: 11: Loss: 9.109550476074219
Step: 12: Loss: 9.001686096191406
Step: 13: Loss: 8.905025482177734
Step: 14: Loss: 8.783361434936523
Step: 15: Loss: 8.669543266296387
Step: 16: Loss: 8.529560089111328
Step: 17: Loss: 8.397661209106445
Step: 18: Loss: 8.2394437789917
Step: 19: Loss: 8.093700408935547
Step: 20: Loss: 7.923843860626221
Step: 21: Loss: 7.7744269371032715
Step: 22: Loss: 7.604264736175537
Step: 23: Loss: 7.45838737487793
Step: 24: Loss: 7.291067123413086
Step: 25: Loss: 7.1424760818481445
Step: 26: Loss: 6.969850540161133
Step: 27: Loss: 6.810494422912598
Step: 28: Loss: 6.6295623779296875
Step: 29: Loss: 6.462369441986084
Step: 30: Loss: 6.2794

# Training Pipeline — Design Principles

> **Config = what I want. Runtime class = what is actually happening.**

## 1. Separate responsibilities

```text
LLMConfig
    → What is the model?

TrainConfig
    → How do I want to train it?

TokenBatchLoader
    → How do I get batches of tokens?

Trainer
    → How do I execute and track training?
```

## 2. `LLMConfig` → Model / Architecture Decisions

If changing it changes **what the model is**, it belongs here.

```python
LLMConfig(
    vocab_size=50_000,
    ctx_len=128,
    d_model=256,
    n_layer=4,
    n_heads=4,
    attention="mha",
    position_embedding="sinusoidal",
)
```

Examples:

* `vocab_size`
* `ctx_len`
* `d_model`
* `n_layer`
* `n_heads`
* MHA / GQA / MQA / MLA
* RoPE / positional encoding
* normalization type
* FFN size

## 3. `TrainConfig` → Experiment / Training Decisions

If changing it changes **how the same model is trained**, it belongs here.

```python
TrainConfig(
    batch_size=32,
    target_tokens=5_000_000,
    learning_rate=3e-4,
    weight_decay=0.01,
    beta1=0.9,
    beta2=0.95,
)
```

Examples:

* `batch_size`
* training token budget / `target_tokens`
* learning rate
* weight decay
* optimizer hyperparameters
* warmup / LR schedule
* gradient clipping
* logging frequency
* checkpoint frequency
* device preference (`"auto"`, `"cuda"`, `"mps"`, etc.)

**Important:** `ctx_len` belongs to `LLMConfig`, while `batch_size` belongs to `TrainConfig`.

```text
ctx_len  → property of the model / input representation
batch    → property of the training procedure
```

Avoid having two sources of truth:

```python
T = 128

LLMConfig(ctx_len=T)
TokenBatchLoader(T=T)
```

Prefer:

```python
llm_config.ctx_len
train_config.batch_size
```

and wire them together when constructing the loader.

## 4. `TokenBatchLoader` → Data Access

It owns things related to **reading and producing token batches**:

```text
binary file
memmap
B, T
current position
next_batch()
```

It should not know about:

* model
* optimizer
* loss
* checkpoints

## 5. `Trainer` → Runtime State + Orchestration

The Trainer owns objects/state that **change while training runs**:

```python
self.model
self.dataloader
self.optimizer
self.device
self.step
self.tokens_seen
```

The Trainer uses `TrainConfig` to decide what to do:

```text
TrainConfig.max/target tokens
        ↓
Trainer.current step / tokens seen
```

So:

```text
CONFIG                     RUNTIME
────────────────────────────────────────
learning_rate       →     optimizer
target_tokens       →     tokens_seen / stopping condition
batch_size          →     dataloader
device="auto"       →     actual torch.device
checkpoint_interval →     checkpoint execution
```

## Core Rule to Remember

> **If it describes the experiment → Config.**
>
> **If it is an object/state needed while executing the experiment → Trainer.**

Don't create extra abstractions unless they solve a real problem. For this project, `LLMConfig + TrainConfig + TokenBatchLoader + Trainer` is already enough.
